# 08 Agent、Subagent与测试角色


## 1. 本篇对应的课程与资料范围

上一篇把重复使用的检查流程保存为 Skills。本篇接着讨论：怎样把专项任务交给有明确职责的 Subagent，并让测试技能、测试专家和质量工程师配合工作。

对应课程文件：

```text
17-Agent-Agent概念及单元测试技能开发.mp4
18-Agent-数字员工测试专家Subagent开发.mp4
19-Agent-数字员工质量工程师Subagent开发.mp4
```

课程见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`。本篇已读取配套 [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p) 的 6.2“子代理（Sub-agent）并行处理”，以及网盘 `项目/黑马记账 APP/` 中的实际技能、代理和 `package.json`。

**资料版本要先区分：**04 所依据的开发讲义采用 Tauri + React + TypeScript、IndexedDB + Dexie；此次网盘项目快照采用 Electron + React + TypeScript，包含 `sql.js`、Zustand 和 Vitest。下面的实际文件和测试命令来自这个 **Electron 项目快照**，不能据此断言前篇 Tauri 示例也已经配置了测试。

视频未逐段观看或转录，不将项目最终快照当成视频每一步的记录。概念图、原始配置和关键流程按资料整理；当前 Claude Code 行为及调用提示词明确标为补充，官方文档核对日期为 `2026-10-10`。本篇只保存学习笔记，不创建或运行实际代理。


## 2. Agent：围绕目标使用工具完成任务

课程将 Agent 称为“代理、智能体”，用“数字员工”比喻有专项职责的 Subagent。图中左侧是技能、记忆与模型，中央是 Claude Code 主代理，右侧是子代理及可复用技能。

<img src="images/08-01-agent-concept.jpg" alt="课程 Agent 概念图：Claude Code 主代理路由调用子代理，子代理结合 Skills 完成任务" width="900">

图片取自网盘 `Agent概念绘图.drawio.png` 的页面预览，保留为本地 JPEG，便于离线阅读。

理解图中的关系，可以从一次测试任务入手：用户给主代理一个目标；主代理判断需要哪类专项工作；测试专家读取相关代码、使用测试技能、执行测试，再把结果交回主代理。模型负责理解与判断，工具负责读文件、改文件或运行命令，技能提供可重复使用的步骤。

**补充：**普通文本回答与 Agent 工作的区别，在于是否围绕目标持续进行“判断下一步 → 使用工具 → 观察结果 → 调整行动”。它并不保证每一步都正确，完成情况仍要用文件改动、命令输出和实际行为验证。

图是概念关系图，不能从中推断所有子代理都有跨会话记忆、自动加载所有技能，或总是同时运行。具体能力由代理定义、当前版本和会话环境共同决定。


## 3. 主 Agent、Subagent 与 Skill 各负责什么

| 对象 | 主要职责 | 本篇中的对应关系 |
|---|---|---|
| 主 Agent | 接收目标、拆分任务、选择执行者、汇总结果 | 当前 Claude Code 主会话 |
| Subagent | 在专项职责和工具范围内完成被委派的任务 | `tester`、`quality-engineer` |
| Skill | 保存可复用的流程、规则和输出要求 | `unit-test`、`security-audit`、`comments-check` |

Skill 说明“这件事怎么做”；Subagent 说明“由谁在什么职责范围内做”。一个角色可以组合多个技能，同一个技能也可以被不同任务复用。

课程项目中的实际调用关系是：

```text
Claude Code 主会话
├── tester
│   └── Skill: /unit-test
└── quality-engineer
    ├── Skill: /security-audit
    ├── Skill: /comments-check
    └── 代理正文中的代码质量检查规则
```

**补充：**自定义 Subagent 通常有自己的上下文窗口、系统提示词和工具配置。详细检索、日志等可以留在子代理上下文中，主会话接收结果摘要；但每个子代理仍会消耗模型请求与 token，并不是没有成本的并行计算。

测试专家产出“哪些行为被测试、结果怎样”；质量工程师产出“代码还有哪些风险及改进点”。通过测试不等于质量审查全部通过，质量审查也不能代替运行测试。


## 4. 讲义原例：独立任务可以并行处理

讲义 6.2 写道：“Claude Code 可以派生子代理并行处理独立任务。”列出的场景包括同时研究多个技术方案、并行审查多个文件、同时搜索不同维度的信息。

原始提示词：

```text
请同时做三件事：
1. 审查 src/api/ 下所有文件的错误处理
2. 审查 src/components/ 下所有组件的性能
3. 审查 src/utils/ 下所有工具函数的测试覆盖
```

随后讲义说明：“Claude 会自动决定哪些任务可以并行处理。”上面的目录属于通用示例，不能据此认定黑马记账项目一定有 `src/api/`。

**补充：**并行适合输入明确、互不依赖的阅读与分析。若质量审查必须基于测试专家刚写完的测试，或者两个角色都会修改同一文件，就要安排先后顺序。上下文独立不等于文件系统隔离；并行任务仍可能访问同一份工作目录。

本篇整理笔记时没有启动这些子代理。这里保留的是课程中的用法。


## 5. 原项目中的文件怎样组织

网盘 `项目/黑马记账 APP/` 中与本篇直接相关的结构如下。树中省略其它文件；`src/utils/` 的文件名已在网盘目录核对，TypeScript 文件正文未通过预览读取。

```text
黑马记账 APP/
├── package.json
├── vitest.config.ts
├── src/
│   ├── test-setup.ts
│   └── utils/
│       ├── finance.ts
│       ├── finance.test.ts
│       ├── category.ts
│       ├── category.test.ts
│       ├── snake-game.ts
│       └── snake-game.test.ts
└── .claude/
    ├── agents/
    │   ├── tester.md
    │   ├── quality-engineer.md
    │   └── gitcommit-agent.md
    ├── skills/
    │   ├── unit-test/SKILL.md
    │   ├── security-audit/SKILL.md
    │   └── comments-check/SKILL.md
    └── results/
```

`gitcommit-agent.md` 和结果目录出现在最终项目快照中，本篇只解释两类测试角色已有的结果交接约定，不展开提交代理和后续 Hook 实现。

**补充：**项目代理位于 `<应用项目>/.claude/agents/<名称>.md`，个人代理位于 `~/.claude/agents/<名称>.md`。项目技能仍使用上一篇的 `.claude/skills/<名称>/SKILL.md`。代理是一个 Markdown 文件，技能是包含 `SKILL.md` 的目录，二者不能混用。

操作这些文件时，工作目录应为含应用 `package.json` 的项目根目录。当前存放 Notebook 的学习笔记目录并不是黑马记账应用根目录。


## 6. 单元测试技能：先确定目标，再创建和运行测试

原文件是 `.claude/skills/unit-test/SKILL.md`，头部如下：

```yaml
---
name: unit-test
description: 创建单元测试、执行测试、并生成测试报告
---
```

它规定使用 **Vitest**，需要模拟浏览器时使用 **jsdom**，测试文件命名为 `*.test.ts` 或 `*.test.tsx`，放在 `src/` 下。

原技能的流程有四步：

1. **确定测试目标。**指定了文件或函数，就分析该目标；未指定时，列出适合测试的模块让用户选择。
2. **创建测试文件。**与被测文件同级，按 Vitest 结构编写测试。
3. **执行测试。**根据任务运行所有测试或指定文件的测试。
4. **生成报告。**根据实际 Vitest 输出整理数量、通过情况和失败明细。

原技能给出的目标优先级：

| 优先级 | 目标 | 原资料的说明 |
|---|---|---|
| 1 | 纯计算函数，如 `calcBalance`、`formatAmount` | 最简单，依赖少，适合先测试 |
| 2 | 数据转换函数，如 `mapCategory` | 逻辑明确，适合验证输入输出 |
| 3 | store 中的状态管理逻辑 | 可能需要 mock 依赖 |
| 4 | React 组件，如弹窗、表单验证 | 通常要处理更多依赖 |

**补充：**单元测试把小范围逻辑放进可重复执行的场景中，比较实际结果与预期结果。mock 是用可控制的替身替代依赖；是否需要 mock 要看真实依赖，不能因为代码属于 store 或组件就机械地全部替换。

测试目标应来自业务约定，不能只把当前实现的输出原样写成“预期值”。


## 7. 原技能的测试文件结构与编写规则

原资料的文件对应关系：

```text
src/utils/finance.ts       → src/utils/finance.test.ts
src/stores/useAppStore.ts  → src/stores/useAppStore.test.ts
src/pages/CategoryPage.tsx → src/pages/CategoryPage.test.tsx
```

原始测试结构如下。`实际值`、`预期值` 和被测对象导入都是待填写内容，这段是结构模板，不是已经完成的可运行测试。

```typescript
import { describe, it, expect } from 'vitest'
// 导入被测函数/组件

describe('模块名 — 功能描述', () => {
  it('场景1：输入 → 预期输出', () => {
    // Arrange（准备数据）
    // Act（执行操作）
    // Assert（验证结果）
    expect(实际值).toBe(预期值)
  })

  it('场景2：边界情况', () => {
    // ...
  })

  it('场景3：异常情况', () => {
    // ...
  })
})
```

`describe` 组织一组相关测试，`it` 描述一个场景，`expect` 表达该场景应满足的结果。Arrange、Act、Assert 对应准备数据、执行操作和验证结果。

原技能要求每个 `it()` 只测一个场景，用中文说明输入和预期输出，并覆盖正常、边界及异常情况，参考 `src/utils/finance.test.ts` 的风格。它还列出常用断言 `toBe`、`toEqual`、`toBeDefined`。

**补充：**0、空值、负数是否属于合法输入，要由被测函数的约定决定。例如“异常情况”可能要求抛错，也可能要求返回约定结果，不能统一断言为同一种处理。`toBe` 适合值或引用比较，结构内容比较通常使用 `toEqual`；断言要能识别目标错误，而不是只检查结果“存在”。


## 8. 实际测试命令：以原项目 package.json 为准

此次读取的网盘项目 `package.json` 中，测试脚本确实存在。以下为测试相关字段的原文摘录：

```json
{
  "scripts": {
    "test": "vitest run",
    "test:watch": "vitest",
    "test:coverage": "vitest run --coverage"
  }
}
```

`devDependencies` 中包含 `vitest`、`jsdom`、`@testing-library/react`、`@testing-library/jest-dom` 和 `@testing-library/user-event`。这说明项目声明了这些测试依赖，不等于本机依赖已经正确安装或测试已经通过。

在**该应用项目根目录且依赖已就绪**的前提下，原技能的命令如下：

| 命令 | 用途 |
|---|---|
| `npm run test` | 按脚本执行所有匹配的测试一次 |
| `npx vitest run` | 直接调用 Vitest，运行一次 |
| `npx vitest run src/utils/finance.test.ts` | 原命令“文件路径”的具体用法，只运行目标文件 |
| `npm run test:watch` | 监听文件变化并重新运行测试 |

例如，先验证目标文件，再检查整体回归：

```bash
npx vitest run src/utils/finance.test.ts
npm run test
```

这是补充的验证顺序，本次没有执行。若目标测试通过而全量测试失败，应分别报告，不能只保留局部通过结果。监听模式持续运行，不能把“进程还没退出”直接当成测试卡死或全量完成。

`test:coverage` 是快照中额外存在的脚本；本篇不展开覆盖率工具配置。脚本存在也不保证相应覆盖率 provider 已安装，更不能凭脚本声明覆盖率数值。

对于前篇 Tauri 项目，仍须重新检查它自己的 `package.json`、测试配置和依赖。不要直接假定同名命令和本快照含义相同。


## 9. 原技能的测试报告：数字必须来自运行结果

原 `unit-test` 技能给出的成功报告模板：

```text
========== 单元测试报告 ==========

📁 测试文件：3 个
📝 测试用例：25 个

✅ 通过：25
❌ 失败：0
⏱️  耗时：960ms

==================================
```

有失败时，原模板要求附加明细：

```text
———— 失败明细 ————

❌ src/utils/finance.test.ts > calcBalance — 收支相等
   期望值：0
   实际值：-0
   建议：检查浮点数处理逻辑

==================================
```

**这些数字和失败案例是技能文件里的示例，不是本次测试结果。**实际报告必须替换为本次运行的文件数、用例数、结果和耗时，并保留跳过、未执行或环境失败等状态。

原例中的 `0` 与 `-0` 提醒读者留意数值边界；“检查浮点数处理逻辑”是排查方向，不能只看模板就认定实际根因。应结合真实函数、断言和输出判断业务是否区分两者。

**补充：**报告中同时写清工作目录、实际命令、目标范围和退出状态。找不到脚本、依赖缺失、没有匹配测试、工具运行中断，都不应写成“测试全部通过”。单元测试也不能单独证明 Electron 窗口、文件导出和本地数据持久化已经正常。


## 10. 测试专家 tester：把测试技能交给专项角色

原文件 `.claude/agents/tester.md` 的普通工作部分如下。最终快照另有 Gate Mode 段落，放到第 18 节解释。

````markdown
---
name: tester
description: 单元测试专家——当用户需要创建单元测试、执行测试、查看测试报告时调用此 subagent。使用项目 /unit-test 技能来完成工作。
model: haiku
tools: Skill, Read, Write, Bash, Glob, Edit, Grep
---

## 你是谁

你是黑马记账项目的**单元测试专家**。你的职责是帮用户创建、执行和管理单元测试。

## 工作方式

当你被调用时，**第一步就是使用 Skill 工具调用 `/unit-test` 技能**。这个技能包含了完整的测试流程指引：

1. **确定测试目标** — 分析代码，找到适合测试的函数/组件
2. **创建测试文件** — 按照 Vitest 规范编写测试
3. **执行测试** — 运行 `npx vitest run`
4. **生成测试报告** — 输出中文格式的测试报告

## 技术栈（项目已配置）

- **测试框架**：Vitest（运行 `npx vitest run`）
- **测试环境**：jsdom（模拟浏览器）
- **测试文件位置**：`src/**/*.test.{ts,tsx}`
- **配置文件**：`vitest.config.ts`

## 重要规则

- 优先测试**纯计算函数**（如 `src/utils/` 下的工具函数）—— 最简单，最有价值
- 每个 `it()` 只测一个场景，用中文描述
- 覆盖三种情况：**正常输入**、**边界情况**（0、空值）、**异常情况**（负数、非法输入）
- 参考已有的测试文件风格（如 `src/utils/finance.test.ts`）
- 测试写完必须立即运行，确认全部通过后才算完成
- 如果测试失败，分析原因并修复，不要留到下次

## 输出

每次完成工作后，以中文格式汇报：
- 新增/修改了哪些测试文件
- 测试用例数量
- 通过/失败情况
- 如果有失败，说明原因和修复方案
````

这里把角色职责和具体流程分开放置：`tester` 管测试任务，`unit-test` 管创建、执行与报告的步骤。更新测试流程时优先更新技能，避免代理正文再维护一套相互冲突的详细步骤。

**补充：**原规则要求分析失败并修复，但修复仍须遵守本次任务范围。失败可能来自业务实现、测试预期或环境，不能为了变绿就删除断言、跳过失败用例或改写未授权的业务行为。


## 11. 读懂代理头部、工具与上下文

以下按当前官方 Subagents 文档解释原配置：

| 配置 | 本例 | 含义 |
|---|---|---|
| `name` | `tester` | 角色标识，用于识别与请求委派 |
| `description` | 创建单元测试、执行测试、查看报告 | 帮助主代理判断何时使用 |
| `model` | `haiku` | 原项目选择的模型别名，具体可用模型受环境影响 |
| `tools` | `Skill, Read, Write, Bash, Glob, Edit, Grep` | 该代理可使用的工具集合 |
| YAML 后的 Markdown | “你是谁”“工作方式”等 | 子代理的系统提示词正文 |

`Skill` 让代理调用技能，`Read`、`Glob`、`Grep` 用于读取与定位代码，`Write`、`Edit` 支持写文件和修改，`Bash` 用于终端命令。工具可用性及具体执行仍受环境和会话权限约束。

原项目没有配置 `skills:` 预加载，而是在正文要求第一步调用 `/unit-test`。当前官方文档允许通过 `Skill` 工具发现并调用项目技能，因此不能把“没有预加载”误判成“无法调用”。

**补充：**若希望启动时就加载技能正文，可在代理 frontmatter 中增加下面的可选字段。这是当前写法的补充，不是原文件已有字段，也不需要与正文重复强制加载：

```yaml
skills:
  - unit-test
```

当前文档说明，设置 `disable-model-invocation: true` 的技能不能被代理预加载。本篇原 `unit-test` 没有该字段，不要机械复制 07 的“仅手动调用 lint 技能”设置，否则会影响测试角色对技能的使用。

普通自定义子代理从自己的上下文开始工作，不自动拥有主会话的完整聊天历史。委派时要传递目标、业务预期、文件范围和已知问题；项目约定可以由相关文件提供。图中的“记忆”也不代表每个角色默认开启了专门的跨会话持久记忆。

`tools` 是子代理的工具允许列表，与 07 技能中的 `allowed-tools` 含义不同。即便不列 `Edit`，只要保留 `Write` 或可写文件的终端工具，也不能宣称这个角色在技术上完全只读。


## 12. 补充：怎样创建、发现和调用 tester

在应用项目的 `.claude/agents/tester.md` 保存定义，同时保证 `.claude/skills/unit-test/SKILL.md` 存在。也可以让 Claude Code 按明确职责生成文件，再打开核对 frontmatter 和正文；模型生成的配置仍需检查。

当前官方文档说明，已存在的代理目录中的文件变化通常会自动加载；首次新建 `agents/` 目录等情况可能需要重启。遇到找不到角色时，先核对工作目录、文件位置、`name` 和来源冲突，再按当前版本重启验证。

旧版 `/agents` 可能打开创建和管理界面；本次核对的当前文档说明，新版会提示直接让 Claude 创建或编辑代理文件。不要把某一版菜单截图当作所有版本的固定操作。

下面是**补充调用示例**，应输入 Claude Code 会话，而不是 PowerShell：

```text
请使用 tester 子代理，为 src/utils/finance.ts 中的目标函数补充单元测试。
先读取项目约定、unit-test 技能和已有 finance.test.ts。
根据函数真实约定覆盖正常、边界和异常场景，只修改本次需要的测试文件。
先运行目标文件测试，再运行项目全量测试。
返回改动文件、实际命令、用例结果、失败原因与尚未验证的范围。
```

需要把“目标函数”和业务预期补充清楚。上面文件名已在网盘核对，但函数实现没有在本次预览读取，不据此编造其参数或测试答案。

自然语言明确角色名通常会促成委派；当前版本也支持在 `@` 选择器选择相应 agent。代理不是自动注册成同名 Skill，不能因为文件叫 `tester.md` 就假定 `/tester` 是有效技能命令。

验证时应在会话记录看到实际委派及其结果，并核对测试文件和终端输出；主代理只说“我按测试专家方式分析了”不能证明自定义子代理已经加载和运行。


## 13. 质量工程师 quality-engineer：组合三类检查

原文件 `.claude/agents/quality-engineer.md` 的头部：

```yaml
---
name: quality-engineer
description: 代码质量工程师——当用户需要对代码进行质量审查、安全检查、注释检查时调用。综合使用 /security-audit 和 /comments-check 技能，并额外检查错误处理、类型安全、重复代码等质量问题。
model: sonnet
tools: Skill, Read, Grep, Glob, Bash, Write
---
```

原角色从三个维度审查代码：

| 维度 | 使用方式 | 检查内容 |
|---|---|---|
| 安全 | 调用 `/security-audit` | 敏感信息、SQL 注入、配置及 Electron 安全 |
| 注释 | 调用 `/comments-check` | 注释比例、准确性、小白可读性 |
| 代码质量 | 使用代理正文中的规则 | 错误处理、类型安全、重复及无用代码 |

原流程先确定范围，再执行安全审计、注释检查和代码质量审查，最后输出综合报告。用户指定文件或目录就检查该范围；未指定则默认检查 `src/` 下最近修改文件或询问用户；“全面检查”对应整个 `src/`。

原文件的原则还提到 `src/`、`electron/` 源码；安全技能的默认范围也包含配置文件。**补充：**组合调用前应统一范围，并将同一范围传给技能，防止一个角色检查局部文件，另一个技能按默认值扩展到全项目。涉及 Electron 安全时，需要明确包含主进程相关文件。

原角色选择 `sonnet`，测试角色选择 `haiku`，体现了按任务配置模型的方式；这不是测试只能使用 Haiku、审查只能使用 Sonnet 的限制。

原原则是普通审查不修改源文件、给出修改前后示例并标注严重程度。虽然工具表保留了 `Bash` 和 `Write`，正文的只读要求仍需遵守；这些工具也说明“默认只读”是任务规则，不能被误解成技术层面没有写入能力。


## 14. 安全审计技能：从搜索线索到核对实际风险

原 `.claude/skills/security-audit/SKILL.md` 的头部：

```yaml
---
name: security-audit
description: 安全审计：检查敏感信息泄露、SQL注入、配置安全等隐患
---
```

原技能按五个维度检查：

| 维度 | 原技能的主要检查项 |
|---|---|
| 敏感信息 | 明文密码、API Key/Token、含密码的连接串、私钥证书、写死的 salt |
| SQL 注入 | 拼接用户输入、参数化查询、LIKE 输入处理、动态标识符的白名单 |
| 配置安全 | `.env` 等是否进入 Git，`settings.json` 等是否含凭据，第三方密钥是否暴露 |
| Electron 安全 | `contextIsolation`、`nodeIntegration`、`webSecurity`，远程内容、生产调试及 `shell.openExternal()` 的 URL 校验 |
| 其它风险 | XSS、`eval`/`new Function`、安全场景中的 `Math.random()`、敏感 localStorage、输入校验、日志泄露和依赖漏洞 |

原技能保留的搜索关键词：

```text
password, secret, token, apiKey, api_key, privateKey,
private_key, salt, passwd, pwd, auth, credential
```

SQL 示例先展示风险写法，再展示参数化查询：

```typescript
// ❌ SQL 注入风险！用户输入直接拼进 SQL
const sql = `SELECT * FROM transactions WHERE category_l1 = '${userInput}'`
```

```typescript
// ✅ 参数化查询，安全
const sql = `SELECT * FROM transactions WHERE category_l1 = ?`
const rows = await db.queryAll(sql, [userInput])
```

以上是原技能的对比片段，需要对应数据库接口才能使用，不是本次改动。值参数化不能替代动态表名、列名或排序字段的白名单；LIKE 中 `%`、`_` 的处理还要结合业务匹配语义判断。

原流程是确定范围 → 搜索并阅读五类代码 → 输出风险分级、位置及修复示例。默认范围是 `src/`、`electron/` 与配置文件，同时检查 `.gitignore`。`npm audit` 是原技能提到的依赖检查入口，本次未运行。

**补充：**搜索命中是线索，不能把变量名 `token`、一个普通 URL 或任何 `Math.random()` 都直接报为漏洞。应核对数据来源、用途、可达路径和实际后果。原文要求结合本地 Electron + SQLite 场景判断；“桌面应用”也不能作为忽略不可信输入、渲染进程风险或本地数据保护的依据。该技能要求只读分析并给出建议。


## 15. 注释检查技能：比例、准确性与可读性

原 `.claude/skills/comments-check/SKILL.md` 的头部：

```yaml
---
name: comments-check
description: 检查代码注释质量：覆盖率、准确性、小白可读性
---
```

原技能提出“每 10 行代码中，至少 3 行是注释”，即注释率至少 30%。计算式为 `注释行数 ÷ 总行数 × 100%`，排除空白行，注释包含 `//`、`/* */`、`/** */` 和 JSDoc。

检查对象包括函数说明、核心变量、关键分支和复杂表达式或算法。另两项是注释与实际代码是否一致，以及读者能否从通俗说明中看懂用途、参数和返回结果。

原技能用下面的片段说明“注释与行为不一致”：

```typescript
// ❌ 注释和代码不匹配
/** 计算本月总支出 */
function calcTotal(transactions) {
  return transactions.filter(t => t.type === 'income')  // 实际只取收入！
    .reduce((sum, t) => sum + t.amount, 0)
}
```

小白可读性的原例：

```typescript
/**
 * 把蛇头往前移动一格
 * 就像蛇往前爬了一步，新位置 = 当前位置 + 方向
 * @param head - 蛇头的当前坐标 {x, y}
 * @param direction - 移动方向 {x: 1=右, -1=左; y: 1=下, -1=上}
 * @returns 蛇头移动后的新位置
 */
function moveSnakeHead(head: Point, direction: Point): Point {
  return { x: head.x + direction.x, y: head.y + direction.y }
}
```

原资料将只写“执行 Point 对象的向量加法运算”作为较难懂的对照。这是说明方式的比较；`Point` 的类型定义未包含在片段中。

原评分标准：覆盖率 ≥30% 为三星，15%—29% 为二星，低于 15% 为一星；准确性看是否匹配实现，可读性看是否通俗且解释充分。流程为确定目标 → 逐项分析 → 输出文件、行数、注释率、问题位置和改进示例，**只出报告，不修改代码**。

**补充：**30% 是本课程项目的教学规则，不是所有项目通用的质量定律。统计时还要明确代码与注释同行等情况如何计数；原文件没有完整定义这些细节，不能把模型估计写成精确测量。比例高也不能抵消过期或误导的注释。


## 16. 质量工程师正文中的额外检查

调用两个技能后，原 `quality-engineer` 还要求检查以下内容。这里的“内置规则”指这个自定义代理正文里的规则，不是 Claude Code 内置的质量检测器。

| 类别 | 原规则 |
|---|---|
| 错误处理 | async/Promise 的错误处理；数据库失败；空值、边界值和非法输入；文件读写错误 |
| 类型安全 | 滥用 `any`；不必要的 `as`；`?.` 和 `??` 是否合理；函数参数和返回类型 |
| 重复代码 | 3 行以上相同逻辑；可提取为工具函数的重复逻辑 |
| 无用代码 | 未使用的 import；注释掉的代码；未调用的函数和变量 |

原角色还要求对问题给出“修改前 → 修改后”的具体对比，标注风险等级，并记录做得好的地方。

**补充：**这些检查项是阅读线索，仍需结合语言行为和项目约定判断。异步错误可能由上层集中处理，不是每个 async 函数都必须在本地写 `try-catch`；TypeScript 可推导类型，不是没有显式返回类型就一定不安全；相似的三行代码也不必自动抽象。

报告应解释真实影响。例如，指出“某条数据库异常路径没有反馈给调用方，界面可能保留过期状态”，比只说“缺少 try-catch”更有用。若某项只能给出建议，应与可复现缺陷分开表述。


## 17. 原质量报告模板与修复优先级

原 `quality-engineer` 的综合报告模板如下，文件名与所有数字均为示例或占位内容：

```text
========== 代码质量综合报告 ==========

📁 审查范围：src/pages/CategoryPage.tsx
📅 审查时间：YYYY-MM-DD HH:MM

====================================
总体评分：XX / 100
====================================

———— 一、安全审计（/security-audit）————

🔴 高危：X 个
🟡 中危：X 个
🟢 低危：X 个

[问题列表，含风险等级、位置、修复建议]

———— 二、注释检查（/comments-check）————

📊 注释率：XX%

⚠️ 缺少注释：X 处
⚠️ 注释不准确：X 处
⚠️ 注释不够小白：X 处

[问题列表，含具体位置和改进示例]

———— 三、代码质量（内置规则）————

🔧 错误处理：X 个问题
🔧 类型安全：X 个问题
🔧 重复代码：X 处
🔧 无用代码：X 处

[问题列表]

====================================
修复优先级：
  1. [最高] 🔴 安全问题
  2. [高]   🔧 错误处理 / 类型安全
  3. [中]   💬 注释补充
  4. [低]   🧹 无用代码清理
====================================
```

原模板没有规定“100 分制”如何计算。**补充：**没有明确评分依据时，不应凭感觉制造精确分数；可以先报告可定位的问题、严重程度和未检查范围，再说明未计算总分。注释率同样需要真实统计口径。

原优先级用于组织修复顺序，具体任务仍按实际影响处理。主代理应核对报告所指代码，再进行授权范围内的修复，而不是直接把所有建议当作已确认缺陷。

调用示例（补充）：

```text
请使用 quality-engineer 子代理审查本次指定的改动文件。
将明确的文件范围传给 security-audit 和 comments-check，不修改源文件。
按原角色规则检查错误处理、类型安全、重复代码和无用代码。
报告具体位置、问题影响、修复示例及未检查范围；没有统计依据的分数或比例不要编造。
```


## 18. 项目快照中的 Gate Mode：角色怎样交接结果

两个代理的最终文件都带有“质量门模式”：当被 `gitcommit-agent` 调用且 prompt 包含“质量门”关键词时，普通工作完成后，最后一步写入结果文件。普通调用只输出对话报告，不写通行证文件。

这说明快照已经包含后续工作流的接点，不能据此认定第 18、19 节视频的初版文件就有全部 Gate Mode 内容。本篇只记录交接格式，不实现 Hook 或提交自动化。

| 角色 | 结果文件 | 原 PASS 条件 |
|---|---|---|
| `tester` | `.claude/results/tester-result.txt` | Vitest 退出码为 0，所有测试通过 |
| `quality-engineer` | `.claude/results/quality-result.txt` | 安全、注释、代码质量三个维度全部通过 |

原文件的目录准备命令是：

```bash
mkdir -p .claude/results
```

如在原生 PowerShell 中手动准备目录，可使用下面的等价补充写法；这里均未执行：

```powershell
New-Item -ItemType Directory -Path '.claude/results' -Force
```

两个角色的通过结果内容均为：

```text
PASS
```

测试专家的失败格式：

```text
FAIL
测试文件：X 个
测试用例：Y 个
通过：Z 个
失败：W 个
失败明细：
  [列出每个失败的测试，含文件路径、用例名、期望值 vs 实际值]
```

质量工程师原文的判定标准：

| 维度 | PASS 条件 | FAIL 条件 |
|---|---|---|
| 安全审计 | 0 个高危 + 0 个中危 | 有高危或中危 |
| 注释检查 | 注释率 ≥20% | 注释率 <20% |
| 代码质量 | 无必须修复的问题 | 有未处理异常或严重类型错误 |

它的失败格式：

```text
FAIL
安全：X 个高危，Y 个中危
注释：XX% 覆盖率（阈值 20%）
代码质量：Z 个必须修复的问题
判定：BLOCKED — 请修复上述问题后再提交
```

**原资料的差异：**`comments-check` 技能和质量工程师前半段要求 30%，Gate Mode 表格写 20%。原文没有进一步解释二者关系，本篇分别保留为检查目标和该快照的门禁阈值；正式使用前应明确项目采用的规则，不能默默合并成同一数字。

**补充：**通行证只是角色间约定的数据文件，不会自行拦截提交。消费结果时还需知道它对应本次哪份代码；旧 `PASS` 不能证明后来修改的代码通过，命令未执行或检查不完整也不能写 PASS。这些结果文件的具体消费机制留给后续章节。


## 19. 补充：主代理怎样把测试与审查接成完整流程

一次功能改动可以按下面的顺序组织。它是依据本篇角色职责整理的实践建议，不是本次实际运行记录。

1. 主代理明确改动范围和预期行为，把必要的业务条件传给角色。
2. `tester` 使用 `unit-test`，补充或执行目标测试，返回真实结果。
3. 在需要审查最终代码时，等测试相关改动完成，再让 `quality-engineer` 审查明确的文件集合。
4. 主代理阅读失败和质量问题，判断属于业务实现、测试、环境还是建议项，处理授权范围内的内容。
5. 对最终改动重新运行受影响的测试，必要时重查相关质量问题，再汇总已完成和未验证部分。

若两个角色只做互不依赖的只读分析，可以并行；若一个角色需要另一个角色的产物，就应顺序执行。工具返回结果后，主代理仍负责判断信息是否充分，不能把两个“完成”直接拼成“项目全部正常”。

| 证据 | 能说明什么 | 仍不能单独证明什么 |
|---|---|---|
| 目标单元测试通过 | 已执行场景的断言满足 | 所有模块和交互都正确 |
| 全量测试通过 | 当前配置收集到的测试通过 | 未写测试的路径也正确 |
| 质量审查无阻塞问题 | 指定范围按给定规则未发现阻塞项 | 没有任何潜在缺陷 |
| 类型与构建检查通过 | 对应类型及构建流程完成 | 桌面程序运行和数据读写全部正常 |
| 指定桌面操作验证 | 这些实际操作符合预期 | 未执行操作同样符合预期 |

写测试和做审查的目的，是给改动增加可检查的依据。最后的结论应覆盖真实执行过的范围。


## 20. 补充：验证角色有没有真正按配置工作

以下是学习时的验证项，未在本次笔记整理中执行：

| 场景 | 应核对的行为 |
|---|---|
| 请求 `tester` 处理明确目标 | 会话出现委派记录，使用 `unit-test`，范围与目标一致 |
| 原应用测试入口存在 | 实际运行命令与 `package.json` 一致，结果可追溯 |
| 项目缺少测试依赖或配置 | 报告前提缺失，不写成通过，不擅自换测试框架 |
| 新测试失败 | 分析真实差异，保留失败证据，不删断言掩盖问题 |
| 请求 `quality-engineer` 只审查文件 | 调用两项技能并检查额外规则，普通审查不改源文件 |
| 指定审查范围 | 下游技能沿用同一范围，不自行扩大 |
| 修改角色或技能后再调用 | 实际读取的是更新后的定义，不依赖旧会话印象 |
| 普通调用 | 返回对话报告，不写 Gate Mode 结果文件 |
| 质量门调用 | 完成检查后写本次结果，不复用旧 PASS |

排查时先区分三类问题：**角色未加载**，核对代理目录、名称、YAML 和当前项目；**技能未调用**，核对技能位置、`Skill` 工具及调用限制；**任务未完成**，核对命令、依赖、退出状态和未检查范围。

项目级代理同名冲突的优先级与 Skills 不完全相同，不能直接套用 07 的技能来源顺序。最容易维护的方式是让本篇示例的文件名、`name` 和职责保持一致，并避免同一作用域出现多个同名定义。


## 21. 来源与本次整理状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：核对第 17—19 节标题，读取 Agent 概念图预览。
- [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)：读取 6.2 子代理并行处理正文与原始三项审查提示词。
- 网盘 `项目/黑马记账 APP/.claude/skills/`：读取 `unit-test/SKILL.md`、`security-audit/SKILL.md`、`comments-check/SKILL.md` 的源码模式全文。
- 同项目 `.claude/agents/`：读取 `tester.md`、`quality-engineer.md` 的源码模式全文，区分普通流程与最终快照中的 Gate Mode。
- 同项目 `package.json`：读取测试脚本与依赖；目录中核对 `vitest.config.ts`、`src/test-setup.ts` 和 `src/utils/` 测试文件。TypeScript 正文未通过当前预览读取，因此没有编造其完整配置、函数实现或测试用例。

当前行为核对：

- [Claude Code：Create custom subagents](https://code.claude.com/docs/en/sub-agents)：项目及个人目录、frontmatter、工具、模型、技能预加载、创建与调用、上下文及独立任务的组织方式。核对日期 `2026-10-10`。

本次新增 08 Notebook 和一张课程概念图的本地预览图片。没有在电脑上安装技能或代理，没有执行 Vitest、依赖安装、构建、提交或推送，也没有修改前篇笔记。文中的命令、PASS/FAIL 和报告数字都是课程内容或明确标注的学习示例，不能当成本次运行结果。
